# Radiative-Convective Equilibrium with DataArray Backend

This notebook demonstrates a radiative-convective equilibrium simulation using the default `DataArray` backend in `climt`. This version uses `xarray` and `pint` for unit handling, which is the original behavior of `climt`.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from datetime import timedelta
from sympl import PlotFunctionMonitor, AdamsBashforth, set_backend
from sympl._core.backend import DataArrayBackend
from climt import (
    SimplePhysics, get_default_state,
    EmanuelConvection, RRTMGShortwave, RRTMGLongwave, SlabSurface
)
from sympl import get_backend

# Set the active backend to DataArray (default)
set_backend(DataArrayBackend())
print(f"Active backend: {type(get_backend()).__name__}")

## Plotting Function

This function uses standard `DataArray` methods like `.to_units()` and `.values`.

In [ ]:
def plot_function(fig, state):
    fig.clf()
    ax = fig.add_subplot(2, 2, 1)
    
    # Standard DataArray unit conversion and values access
    ax.plot(
        state['air_temperature_tendency_from_convection'].to_units(
            'degK day^-1').values.flatten(),
        state['air_pressure'].to_units('mbar').values.flatten(), '-o')
    
    ax.set_title('Conv. heating rate')
    ax.set_xlabel('K/day')
    ax.set_ylabel('millibar')
    ax.grid()
    ax.invert_yaxis()

    ax = fig.add_subplot(2, 2, 2)
    ax.plot(
        state['air_temperature'].values.flatten(),
        state['air_pressure'].to_units('mbar').values.flatten(), '-o')
    ax.set_title('Air temperature')
    ax.set_xlabel('K')
    ax.grid()
    ax.invert_yaxis()

    ax = fig.add_subplot(2, 2, 3)
    ax.plot(
        state['air_temperature_tendency_from_longwave'].values.flatten(),
        state['air_pressure'].to_units('mbar').values.flatten(), '-o',
        label='LW')
    ax.plot(
        state['air_temperature_tendency_from_shortwave'].values.flatten(),
        state['air_pressure'].to_units('mbar').values.flatten(), '-o',
        label='SW')
    ax.set_title('LW and SW Heating rates')
    ax.legend()
    ax.set_xlabel('K/day')
    ax.grid()
    ax.set_ylabel('millibar')
    ax.invert_yaxis()

    ax = fig.add_subplot(2, 2, 4)
    net_flux = (state['upwelling_longwave_flux_in_air'] + 
                state['upwelling_shortwave_flux_in_air'] - 
                state['downwelling_longwave_flux_in_air'] - 
                state['downwelling_shortwave_flux_in_air'])
    
    ax.plot(
        net_flux.values.flatten(),
        state['air_pressure_on_interface_levels'].to_units(
            'mbar').values.flatten(), '-o')
    
    ax.set_title('Net Flux')
    ax.set_xlabel('W/m^2')
    ax.grid()
    ax.invert_yaxis()
    
    fig.tight_layout()
    fig.canvas.draw()

## Simulation Setup

In [ ]:
%matplotlib widget

timestep = timedelta(minutes=5)

convection = EmanuelConvectionPy()
radiation_sw = RRTMGShortwave()
radiation_lw = RRTMGLongwave()
slab = SlabSurface()
simple_physics = SimplePhysics()

state = get_default_state([simple_physics, convection, radiation_lw, radiation_sw, slab])

state['air_temperature'].values[:] = 270
state['surface_albedo_for_direct_shortwave'].values[:] = 0.5
state['surface_albedo_for_direct_near_infrared'].values[:] = 0.5
state['surface_albedo_for_diffuse_shortwave'].values[:] = 0.5

state['zenith_angle'].values[:] = np.pi/2.5
state['surface_temperature'].values[:] = 300.
state['ocean_mixed_layer_thickness'].values[:] = 5
state['area_type'].values[:] = 'sea'

time_stepper = AdamsBashforth([convection, radiation_lw, radiation_sw, slab])
monitor = PlotFunctionMonitor(plot_function)

## Run Simulation

The plot below will refresh every 20 iterations. Notice the performance compared to the `unyt` version.

In [ ]:
for i in range(2000):
    convection.current_time_step = timestep
    diagnostics, state = time_stepper(state, timestep)
    state.update(diagnostics)
    
    diagnostics, new_state = simple_physics(state, timestep)
    state.update(diagnostics)
    state.update(new_state)
    
    if (i+1) % 20 == 0:
        monitor.store(state)
    
    state['time'] += timestep
    state['eastward_wind'].values[:] = 3.0